# LAB 01 — From Text Processing to Search

Parts A–C are in calculations.md and prediction.md. This notebook follows Parts D–J from W1.pdf.

In [ ]:
# Google Colab setup: clone the repository and use it as the working directory.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/thangSy221105/NLP_exercise.git"
REPO_DIR = Path("/content/NLP_exercise")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print(f"Repository already exists: {REPO_DIR}")
    os.chdir(REPO_DIR)
    print(f"Working directory: {Path.cwd()}")
else:
    print("Colab clone cell skipped outside Google Colab.")

# 7. Part D — Experiment 1: Inspect the Sparse Representation

## 7.1. Dataset

Use the provided corpus of approximately 30,000 documents. Do not change the dataset for this experiment.

The repository contains c4-train.00000-of-01024-30K.json.gz under lab01/data/. It is a gzip-compressed JSON Lines file with a text field.

In [ ]:
import gzip
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("lab01/data/c4-train.00000-of-01024-30K.json.gz")
DATA_AVAILABLE = False
documents = []
document_ids = []
records = []

if not DATA_PATH.is_file():
    print(f"Dataset not found at {DATA_PATH}.")
else:
    with gzip.open(DATA_PATH, "rt", encoding="utf-8") as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    documents = [record["text"] for record in records]
    document_ids = list(range(len(documents)))
    DATA_AVAILABLE = True
    print(f"Loaded {len(documents):,} documents from {DATA_PATH}.")
    print(f"Empty documents: {sum(not text.strip() for text in documents):,}")
    display(pd.DataFrame(records[:3]))

## 7.2. Xây dựng pipeline

Raw documents
↓
Tokenizer
↓
CountVectorizer
↓
TF
↓
IDF
↓
TF-IDF matrix

LAB conventions: TF = count / total terms, IDF = log(N / df), TF-IDF = TF × IDF. Keep the matrix sparse.

In [ ]:
if DATA_AVAILABLE:
    try:
        from scipy import sparse
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.preprocessing import normalize
        SKLEARN_AVAILABLE = True
    except ImportError as error:
        SKLEARN_AVAILABLE = False
        print(f"Experiment dependencies are unavailable: {error}")
else:
    SKLEARN_AVAILABLE = False

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    vectorizer = CountVectorizer()
    count_matrix = vectorizer.fit_transform(documents)
    tf_matrix = normalize(count_matrix, norm="l1", axis=1, copy=True)
    document_frequency = np.asarray(count_matrix.getnnz(axis=0)).ravel()
    number_of_documents = count_matrix.shape[0]
    idf_values = np.log(number_of_documents / document_frequency)
    tfidf_matrix = tf_matrix.multiply(idf_values).tocsr()
    tfidf_matrix.eliminate_zeros()
    feature_names = vectorizer.get_feature_names_out()
else:
    print("Pipeline skipped because the dataset is not configured.")

## 7.3. Kiểm tra kích thước

In [ ]:
if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    print(f"Number of documents = {count_matrix.shape[0]}")
    print(f"Vocabulary size = {count_matrix.shape[1]}")
    print(f"Matrix shape = {tfidf_matrix.shape}")
else:
    print("Size check skipped because the dataset is not configured.")

### Student answer

> N = TODO
>
> V = TODO

## 7.4. Kiểm tra sparsity

S = 1 - nnz(X) / (N × V)

In [ ]:
if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    nnz = tfidf_matrix.nnz
    total_entries = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
    sparsity = 1 - nnz / total_entries if total_entries else 0.0
    print(f"nnz = {nnz}")
    print(f"Total entries = {total_entries}")
    print(f"Sparsity = {sparsity}")
    print(f"Count matrix sparse = {sparse.issparse(count_matrix)}")
    print(f"TF matrix sparse = {sparse.issparse(tf_matrix)}")
    print(f"TF-IDF matrix sparse = {sparse.issparse(tfidf_matrix)}")
else:
    print("Sparsity check skipped because the dataset is not configured.")

**Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều V?**

### Student answer

> TODO: Viết giải thích ở đây.

## 7.5. Inspect vocabulary

### 7.5.1. Top 20 terms theo document frequency

In [ ]:
def get_top_df_terms(document_frequency, feature_names, top_k=20):
    table = pd.DataFrame({"term": feature_names, "document_frequency": document_frequency})
    return table.sort_values("document_frequency", ascending=False).head(top_k).reset_index(drop=True)

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    display(get_top_df_terms(document_frequency, feature_names))
else:
    print("Document-frequency inspection skipped because the dataset is not configured.")

### 7.5.2. Top 20 terms có IDF cao nhất

In [ ]:
def get_top_idf_terms(feature_names, idf_values, top_k=20):
    table = pd.DataFrame({"term": feature_names, "idf": idf_values})
    return table.sort_values("idf", ascending=False).head(top_k).reset_index(drop=True)

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    display(get_top_idf_terms(feature_names, idf_values))
else:
    print("IDF inspection skipped because the dataset is not configured.")

### 7.5.3. Top 20 terms có TF-IDF cao nhất trong một document được chọn

In [ ]:
def get_top_tfidf_terms(tfidf_matrix, feature_names, document_index, top_k=20):
    row = tfidf_matrix.getrow(document_index)
    table = pd.DataFrame({"term": feature_names[row.indices], "tfidf": row.data})
    return table.sort_values("tfidf", ascending=False).head(top_k).reset_index(drop=True)

SELECTED_DOC_INDEX = 0
if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    print(f"Selected document preview: {documents[SELECTED_DOC_INDEX][:500]}")
    display(get_top_tfidf_terms(tfidf_matrix, feature_names, SELECTED_DOC_INDEX))
else:
    print("Document TF-IDF inspection skipped because the dataset is not configured.")

### 7.5.4. So sánh ba danh sách

### Student answer

> TODO: So sánh ba danh sách.

**Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?**

### Student answer

> TODO

**Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?**

### Student answer

> TODO

# 8. Part E — Core Implementation

## 8.1. Mục tiêu

Tự xây một phiên bản TF-IDF tối giản trên một corpus nhỏ. Không sử dụng trực tiếp TfidfVectorizer cho phần implementation cốt lõi.

## 8.2. Các hàm cần xây dựng

- build_vocabulary()
- compute_counts()
- compute_tf()
- compute_idf()
- compute_tfidf()
- cosine_similarity()

## 8.3. Corpus kiểm thử

- D1 = cat eats fish
- D2 = dog eats fish
- D3 = cat likes fish

## 8.4. Unit tests

Unit tests nằm trong implementation.py. Mỗi hàm cần có ít nhất một test.

## 8.5. So sánh với thư viện

### Student comparison

> TODO: Nếu có khác biệt, xác định do công thức, convention, vocabulary ordering, normalization hay implementation.

# 9. Part F — Experiment 2: Preprocessing Ablation

## 9.1. Pipeline A — Minimal

Raw text
↓
Lowercase
↓
Tokenization

In [ ]:
import re

def tokenize_with_punctuation(text: str) -> list[str]:
    return re.findall(r"\b\w+\b|[^\w\s]", str(text).lower(), flags=re.UNICODE)

def preprocess_a(text: str) -> list[str]:
    return tokenize_with_punctuation(text)

## 9.2. Pipeline B — Normalized

Raw text
↓
Lowercase
↓
Punctuation normalization
↓
Tokenization
↓
Stopword handling

In [ ]:
STOPWORDS = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "it", "of", "on", "or", "that", "the", "this", "to", "was", "were", "with"}

def tokenize_words(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", str(text).lower(), flags=re.UNICODE)

def preprocess_b(text: str) -> list[str]:
    return [token for token in tokenize_words(text) if token not in STOPWORDS]

## 9.3. Pipeline C — Extended

Raw text
↓
Normalization
↓
Subword tokenization

This notebook uses deterministic character trigrams with boundary markers as its lightweight subword implementation.

In [ ]:
def preprocess_c(text: str) -> list[str]:
    subwords = []
    for word in tokenize_words(text):
        if len(word) <= 3:
            subwords.append(word)
        else:
            bounded_word = f"<{word}>"
            subwords.extend(bounded_word[i:i + 3] for i in range(len(bounded_word) - 2))
    return subwords

## 9.4. So sánh

| Metric | Pipeline A | Pipeline B | Pipeline C |
|---|---:|---:|---:|
| Vocabulary size | TODO | TODO | TODO |
| Average tokens/document | TODO | TODO | TODO |
| Matrix sparsity | TODO | TODO | TODO |
| OOV rate | TODO | TODO | TODO |
| Search performance | TODO | TODO | TODO |

In [ ]:
def identity_analyzer(tokens: list[str]) -> list[str]:
    return tokens

def make_token_vectorizer():
    if not SKLEARN_AVAILABLE:
        raise RuntimeError("scikit-learn is required for vectorization.")
    return CountVectorizer(analyzer=identity_analyzer, lowercase=False, preprocessor=None, tokenizer=None, token_pattern=None)

def fit_tfidf_model(tokenized_documents, raw_documents, preprocess, document_ids=None):
    if not tokenized_documents or not any(tokenized_documents):
        raise ValueError("Corpus is empty after preprocessing.")
    vectorizer = make_token_vectorizer()
    count_matrix = vectorizer.fit_transform(tokenized_documents)
    tf_matrix = normalize(count_matrix, norm="l1", axis=1, copy=True)
    document_frequency = np.asarray(count_matrix.getnnz(axis=0)).ravel()
    idf_values = np.log(count_matrix.shape[0] / document_frequency)
    raw_tfidf_matrix = tf_matrix.multiply(idf_values).tocsr()
    raw_tfidf_matrix.eliminate_zeros()
    search_matrix = normalize(raw_tfidf_matrix, norm="l2", axis=1, copy=True)
    ids = list(range(len(raw_documents))) if document_ids is None else list(document_ids)
    if len(ids) != len(raw_documents):
        raise ValueError("document_ids must match raw_documents.")
    return {
        "vectorizer": vectorizer, "count_matrix": count_matrix, "tf_matrix": tf_matrix,
        "raw_tfidf_matrix": raw_tfidf_matrix, "tfidf_matrix": search_matrix,
        "idf_values": idf_values, "feature_names": vectorizer.get_feature_names_out(),
        "tokenized_documents": tokenized_documents, "raw_documents": list(raw_documents),
        "document_ids": ids, "id_to_index": {doc_id: i for i, doc_id in enumerate(ids)},
        "preprocess": preprocess,
    }

def compute_oov_rate(model, queries):
    vocabulary = set(model["feature_names"])
    total = sum(len(model["preprocess"](query)) for query in queries)
    oov = sum(
        token not in vocabulary
        for query in queries
        for token in model["preprocess"](query)
    )
    return oov / total if total else float("nan")

def precision_at_k(retrieved_ids, relevant_ids, k=5):
    if k <= 0:
        return 0.0
    relevant = set(relevant_ids)
    return sum(doc_id in relevant for doc_id in list(retrieved_ids)[:k]) / k

def recall_at_k(retrieved_ids, relevant_ids, k=5):
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0
    return len(set(list(retrieved_ids)[:k]) & relevant) / len(relevant)

def reciprocal_rank(retrieved_ids, relevant_ids):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def mean_reciprocal_rank(retrieved_by_query, evaluation_set):
    if not evaluation_set:
        return float("nan")
    return float(np.mean([
        reciprocal_rank(retrieved_by_query.get(query, []), relevant_ids)
        for query, relevant_ids in evaluation_set.items()
    ]))

def evaluate_pipeline(raw_documents, preprocess, queries, evaluation_set, search_function=None):
    tokenized_documents = [preprocess(doc) for doc in raw_documents]
    model = fit_tfidf_model(tokenized_documents, raw_documents, preprocess)
    matrix = model["raw_tfidf_matrix"]
    rows, columns = matrix.shape
    total_entries = rows * columns
    metrics = {
        "Vocabulary size": columns,
        "Average tokens/document": float(np.mean([len(tokens) for tokens in tokenized_documents])),
        "Matrix sparsity": 1 - matrix.nnz / total_entries if total_entries else 0.0,
        "OOV rate": compute_oov_rate(model, queries),
        "Search performance": float("nan"),
    }
    if evaluation_set and search_function is not None:
        retrieved = {
            query: search_function(query, model, top_k=5)["Document ID"].tolist()
            for query in evaluation_set
        }
        metrics["Search performance"] = mean_reciprocal_rank(retrieved, evaluation_set)
    return model, metrics

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    pipeline_specs = {"Pipeline A": preprocess_a, "Pipeline B": preprocess_b, "Pipeline C": preprocess_c}
    pipeline_models, pipeline_metrics = {}, {}
    for name, preprocess in pipeline_specs.items():
        pipeline_models[name], pipeline_metrics[name] = evaluate_pipeline(
            documents, preprocess, [], {}
        )
    comparison = pd.DataFrame(pipeline_metrics).reindex(["Vocabulary size", "Average tokens/document", "Matrix sparsity", "OOV rate", "Search performance"])
    display(comparison)
else:
    print("Pipeline comparison skipped because the dataset is not configured.")

## 9.5. Câu hỏi phân tích

1. Lowercasing làm thay đổi vocabulary như thế nào?

### Student answer

> TODO

2. Stopword removal có luôn cải thiện representation không?

### Student answer

> TODO

3. Việc loại punctuation có thể làm mất thông tin gì?

### Student answer

> TODO

4. Pipeline nào tạo ra sparse matrix nhất?

### Student answer

> TODO

5. Pipeline nào cho search tốt nhất?

### Student answer

> TODO

6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?

### Student answer

> TODO

# 10. Part G — Application: Build a Document Search Engine

## 10.1. Bài toán

Input:
User query

Output:
Top-K relevant documents

## 10.2. Pipeline

30K Documents
↓
TF-IDF Index
↓
User Query
↓
TF-IDF Query Vector
↓
Cosine Similarity
↓
Ranking
↓
Top-K Documents

In [ ]:
SEARCH_COLUMNS = ["Rank", "Document ID", "Similarity", "Document preview"]

def empty_search_results():
    return pd.DataFrame(columns=SEARCH_COLUMNS)

def search(query: str, model: dict, top_k: int = 5):
    if top_k <= 0 or not str(query).strip():
        return empty_search_results()
    query_tokens = model["preprocess"](query)
    if not query_tokens:
        return empty_search_results()
    query_counts = model["vectorizer"].transform([query_tokens])
    query_tf = normalize(query_counts, norm="l1", axis=1, copy=True)
    query_tfidf = query_tf.multiply(model["idf_values"]).tocsr()
    query_tfidf.eliminate_zeros()
    if query_tfidf.nnz == 0:
        return empty_search_results()
    query_vector = normalize(query_tfidf, norm="l2", axis=1, copy=True)
    scores = model["tfidf_matrix"].dot(query_vector.T).toarray().ravel()
    ranking = np.argsort(-scores, kind="stable")[:min(top_k, len(scores))]
    return pd.DataFrame([
        {
            "Rank": rank,
            "Document ID": model["document_ids"][index],
            "Similarity": float(scores[index]),
            "Document preview": model["raw_documents"][index][:300],
        }
        for rank, index in enumerate(ranking, start=1)
    ], columns=SEARCH_COLUMNS)

## 10.3. Query examples

- medical image classification
- transformer language model
- deep learning healthcare
- natural language processing

## 10.4. Kết quả cần hiển thị

For each query display:

Rank | Document ID | Similarity | Document preview

In [ ]:
EXAMPLE_QUERIES = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
]

if DATA_AVAILABLE and SKLEARN_AVAILABLE and "pipeline_models" in globals():
    selected_model = pipeline_models["Pipeline A"]
    for query in EXAMPLE_QUERIES:
        print(f"Query: {query}")
        display(search(query, selected_model, top_k=5))
else:
    print("Search display skipped because the dataset is not configured.")

# 11. Part H — Evaluation

## 11.1. Tạo evaluation set

Prepare approximately 5–10 queries with student-provided relevance labels.

In [ ]:
evaluation_set = {
    # TODO: student provides 5–10 queries and relevance labels
}
print("Add relevance labels to evaluation_set before running evaluation.")

## 11.2. Precision@K

P@K = number of relevant documents retrieved / K

In [ ]:
if DATA_AVAILABLE and SKLEARN_AVAILABLE and evaluation_set:
    print("Use precision_at_k(retrieved_ids, relevant_ids, k=5).")
else:
    print("Precision@5 skipped: evaluation_set is empty or dataset is unavailable.")

## 11.3. Recall@K

R@K = number of relevant documents retrieved / number of relevant documents

In [ ]:
if DATA_AVAILABLE and SKLEARN_AVAILABLE and evaluation_set:
    print("Use recall_at_k(retrieved_ids, relevant_ids, k=5).")
else:
    print("Recall@5 skipped: evaluation_set is empty or dataset is unavailable.")

## 11.4. Mean Reciprocal Rank

MRR = (1 / |Q|) × sum_q (1 / r_q)

In [ ]:
if DATA_AVAILABLE and SKLEARN_AVAILABLE and evaluation_set:
    evaluation_rows = []
    selected_model = pipeline_models["Pipeline A"]
    retrieved_by_query = {}
    for query, relevant_ids in evaluation_set.items():
        result_table = search(query, selected_model, top_k=5)
        retrieved_ids = result_table["Document ID"].tolist()
        retrieved_by_query[query] = retrieved_ids
        evaluation_rows.append({
            "Query": query,
            "Precision@5": precision_at_k(retrieved_ids, relevant_ids, 5),
            "Recall@5": recall_at_k(retrieved_ids, relevant_ids, 5),
            "Reciprocal Rank": reciprocal_rank(retrieved_ids, relevant_ids),
        })
    evaluation_table = pd.DataFrame(evaluation_rows)
    display(evaluation_table)
    print(f"MRR = {mean_reciprocal_rank(retrieved_by_query, evaluation_set)}")
    pipeline_metrics = {}
    for name, preprocess in pipeline_specs.items():
        _, pipeline_metrics[name] = evaluate_pipeline(
            documents, preprocess, EXAMPLE_QUERIES, evaluation_set, search
        )
    display(pd.DataFrame(pipeline_metrics).reindex([
        "Vocabulary size", "Average tokens/document", "Matrix sparsity",
        "OOV rate", "Search performance"
    ]))
else:
    print("Evaluation skipped: configure the dataset and evaluation_set first.")

### Results export

Export only after the student supplies relevance labels.

In [ ]:
RESULT_COLUMNS = ["query", "rank", "document_id", "similarity", "relevant"]

def build_results_table(query_results, evaluation_set):
    rows = []
    for query, result_table in query_results.items():
        relevant_ids = set(evaluation_set.get(query, []))
        for result in result_table.itertuples(index=False):
            rows.append({
                "query": query,
                "rank": result.Rank,
                "document_id": result.Document_ID,
                "similarity": result.Similarity,
                "relevant": result.Document_ID in relevant_ids,
            })
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)

def results_path():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "lab01").is_dir():
            return candidate / "lab01" / "results.csv"
    return Path("results.csv")

if DATA_AVAILABLE and SKLEARN_AVAILABLE and evaluation_set:
    selected_model = pipeline_models["Pipeline A"]
    query_results = {query: search(query, selected_model, top_k=5) for query in evaluation_set}
    results_df = build_results_table(query_results, evaluation_set)
    results_df.to_csv(results_path(), index=False)
    display(results_df)
else:
    print("Results export skipped: relevance labels and a fitted model are required.")

# 12. Part I — Error Analysis

Select 2 queries with good results and 2 queries with poor results.

In [ ]:
def inspect_query(query: str, model: dict, expected_relevant=None, top_k: int = 5):
    result_table = search(query, model, top_k=top_k)
    query_terms = set(model["preprocess"](query))
    overlaps = []
    for result in result_table.itertuples(index=False):
        index = model["id_to_index"][getattr(result, "Document_ID")]
        document_terms = set(model["tokenized_documents"][index])
        overlaps.append(len(query_terms & document_terms))
    if not result_table.empty:
        result_table = result_table.copy()
        result_table["Lexical overlap"] = overlaps
    return {
        "Query": query,
        "Expected relevant documents": None if expected_relevant is None else list(expected_relevant),
        "Retrieved documents": result_table,
    }

### Good query 1

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

### Good query 2

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

### Poor query 1

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

### Poor query 2

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

## Failure case quan trọng nhất

### Student analysis

> TODO

# 13. Part J — From Failure to the Next NLP Representation

TF-IDF
↓
Distributional representation
↓
Word Embedding
↓
Contextual Embedding
↓
Transformer

**Làm thế nào để biểu diễn được similarity về nghĩa thay vì chỉ similarity về từ?**

### Student hypothesis

> TODO: Đưa ra ít nhất một hypothesis.